# Find duplicate object keys in the structured metadata table
 
Query the `structured_metadata` table to identify S3 keys that have multiple versions with different hashes. This will often happen for real-time data where a file with the same hame is uploaded repeatedly with different content (e.g. another days' data).

In [ ]:
import os

from data_index.iceberg_config import S3TablesCatalogConfig, IcebergTableConfig

os.environ["AWS_PROFILE"] = "edge-admin"

In [ ]:
# --- Sink config ---
data_index_catalog_config = S3TablesCatalogConfig(
    region="ap-southeast-2",
    arn="arn:aws:s3tables:ap-southeast-2:704910415367:bucket/data-index",
)

structured_metadata_table_config = IcebergTableConfig(
    catalog_config=data_index_catalog_config,
    namespace="data_index",
    table_name=f"structured_metadata_v5",
)

In [ ]:
structured_metadata = structured_metadata_table_config.load()
con = structured_metadata.scan().to_duckdb(table_name="structured_metadata")

con.execute("""
    SELECT COUNT(*)
    FROM structured_metadata
"""
).fetchone()

## Duplicate entries for moorings

In [ ]:
con.execute("""
    SELECT key,
           count(*) as n_files,
           count(distinct version_id) as n_versions,
           count(distinct hash) as n_hashes,
           date(min(time_coverage_start)) as date_start,
           date(max(time_coverage_end)) as date_end,
           date(min(date_created)) as first_created,
           date(max(date_created)) as last_created
    FROM structured_metadata
    WHERE key LIKE 'IMOS/ANMN/%'
    GROUP BY key
    HAVING n_hashes > 1
    ORDER BY key
"""
).fetchdf()